# TP04: AST Traversals with `Lark`

## Grammar Engineering

## MEI/2025-26

#### Nuno Macedo
Universidade do Minho

The following exercises are intended to be solved with pen and paper, and then implemented using a parser generator module for Python, [Lark](https://lark-parser.readthedocs.io/).

# AGs with `Lark` `Transformers`

- AGs consider both synthesized (bottom-up) and inherited (attributes)

- `Ply` (which uses LR parsing) and `Lark` `Transformers` act bottom-up
  - They can encode inherited attributes, but not in a natural way

## Example: Simple arithmetic expressions with declared variables

In [ ]:
from lark import Lark, Transformer

def run_parser(data):
  """
  Lark parser with doctests.

  >>> run_parser("10 + 20")
  Tree(Token('RULE', 'start'), [Tree(Token('RULE', 'decls'), []), Tree(Token('RULE', 'expr'), [Token('INT', '10'), Token('PLUS', '+'), Token('INT', '20')])])

  >>> run_parser("let x = 10; 10 * x")
  Tree(Token('RULE', 'start'), [Tree(Token('RULE', 'decls'), [Tree(Token('RULE', 'decl'), [Tree(Token('RULE', 'id_decl'), [Token('ID', 'x')]), Token('INT', '10')])]), Tree(Token('RULE', 'term'), [Token('INT', '10'), Token('TIMES', '*'), Tree(Token('RULE', 'id_call'), [Token('ID', 'x')])])])

  >>> run_parser("let x = 10; let y = 20 * x; 10 + y * x")
  Tree(Token('RULE', 'start'), [Tree(Token('RULE', 'decls'), [Tree(Token('RULE', 'decl'), [Tree(Token('RULE', 'id_decl'), [Token('ID', 'x')]), Token('INT', '10')]), Tree(Token('RULE', 'decl'), [Tree(Token('RULE', 'id_decl'), [Token('ID', 'y')]), Tree(Token('RULE', 'term'), [Token('INT', '20'), Token('TIMES', '*'), Tree(Token('RULE', 'id_call'), [Token('ID', 'x')])])])]), Tree(Token('RULE', 'expr'), [Token('INT', '10'), Token('PLUS', '+'), Tree(Token('RULE', 'term'), [Tree(Token('RULE', 'id_call'), [Token('ID', 'y')]), Token('TIMES', '*'), Tree(Token('RULE', 'id_call'), [Token('ID', 'x')])])])])

  >>> run_parser("let x = 10; let y = 20 * x; (10 + y) * -x")
  Tree(Token('RULE', 'start'), [Tree(Token('RULE', 'decls'), [Tree(Token('RULE', 'decl'), [Tree(Token('RULE', 'id_decl'), [Token('ID', 'x')]), Token('INT', '10')]), Tree(Token('RULE', 'decl'), [Tree(Token('RULE', 'id_decl'), [Token('ID', 'y')]), Tree(Token('RULE', 'term'), [Token('INT', '20'), Token('TIMES', '*'), Tree(Token('RULE', 'id_call'), [Token('ID', 'x')])])])]), Tree(Token('RULE', 'term'), [Tree(Token('RULE', 'expr'), [Token('INT', '10'), Token('PLUS', '+'), Tree(Token('RULE', 'id_call'), [Token('ID', 'y')])]), Token('TIMES', '*'), Tree(Token('RULE', 'factor'), [Token('MINUS', '-'), Tree(Token('RULE', 'id_call'), [Token('ID', 'x')])])])])
  """
  grammar = r"""

  ?start : decls expr

  decls : decl*

  decl : "let" id_decl "=" expr ";"

  id_decl : ID

  ?expr : term ((PLUS|MINUS) term)*

  ?term : factor ((TIMES|DIV) factor)*

  ?factor : MINUS* literal

  ?literal : id_call | INT | "(" expr ")"

  id_call : ID

  PLUS : "+"
  MINUS : "-"
  TIMES : "*"
  DIV : "/"

  ID : /[a-z]+/
  INT : /\d+/

  %ignore /[ \t]+/
  """

  parser = Lark(grammar)
  return parser.parse(data)

import doctest
doctest.run_docstring_examples(run_parser, globals())

## Version with shared state

- Create a symbol table and update it accordingly

- Will work if left-to-right traversal

- Must distinguish between `ID` declarations and `ID` calls

- Notice the use of EBNF and nodes with arbitrary number of children

- To do: better error handling

In [ ]:
def validate(data):
  """
  Lark validator with doctests.

  >>> print(validate("10 + 20"))
  30

  >>> print(validate("let x = 10; x + 20"))
  30

  >>> print(validate("let x = 10; let y = 20; x + 20 * y"))
  410

  >>> print(validate("let x = 10; x + 20 * y"))
  Undeclared variable y

  >>> print(validate("let x = z + 10; x + 20"))
  Undeclared variable z

  >>> print(validate("let x = x; x + 20"))
  Undeclared variable x

  >>> print(validate("let x = 10; let y = 20; x + 40 + y + 50"))
  120
  """
  class ValidateTransformer(Transformer):
      def __init__(self):
        self.env = {}

      def start(self, items):
        return items[1]

      def decls(self, items):
        pass

      def id_decl(self, items):
        return items[0]

      def decl(self, items):
        self.env[items[0].value] = items[1]
        pass

      def expr(self, items):
        for i in range(0,len(items),2):
          if type(items[i]) is str:
            return items[i]

        res = items[0]
        for i in range(1,len(items)-1,2):

          if items[i].type == "PLUS":
            res += items[i+1]
          elif items[i].type == "MINUS":
            res -= items[i+1]
        return res

      def term(self, items):
        for i in range(0,len(items),2):
          if type(items[i]) is str:
            return items[i]

        res = items[0]
        for i in range(1,len(items)-1,2):

          if items[i].type == "DIV":
            res /= items[i+1]
          elif items[i].type == "TIMES":
            res *= items[i+1]
        return res

      def literal(self, items):
        if type(items[-1]) is str:
          return items[1]
        return -items[1]

      def id_call(self, items):
        if items[0].value not in self.env:
          return "Undeclared variable " + items[0].value
        return self.env[items[0].value]

      def INT(self, items):
        return int(items)

  return ValidateTransformer().transform(run_parser(data))

import doctest
doctest.run_docstring_examples(validate, globals())

## Version with Python closures

- The synthesized values are functions waiting for an environment to evaluate

- Cleaner information flow, no shared state, parent nodes resolve pending evaluations

- Code more cluttered, must propagate function declarations

In [ ]:
def validate(data):
  """
  Lark validator with doctests.

  >>> print(validate("10 + 20"))
  30

  >>> print(validate("let x = 10; x + 20"))
  30

  >>> print(validate("let x = 10; let y = 20; x + 20 * y"))
  410

  >>> print(validate("let x = 10; x + 20 * y"))
  Undeclared variable y

  >>> print(validate("let x = z + 10; x + 20"))
  Undeclared variable z

  >>> print(validate("let x = x; x + 20"))
  Undeclared variable x

  >>> print(validate("let x = 10; let y = 20; x + 40 + y + 50"))
  120
  """
  class ValidateTransformer(Transformer):
      def start(self, items):
        return items[1](items[0])

      def decls(self, items):
        res = {}
        for i,j in items:

          res[i] = j(res)
        return res

      def id_decl(self, items):
        return items[0]

      def decl(self, items):
        return items[0].value, items[1]

      def expr(self, items):
        def eval(env):
          for i in range(0,len(items),2):
            if type(items[i](env)) is str:
              return items[i](env)

          res = items[0](env)
          for i in range(1,len(items)-1,2):

            if items[i].type == "PLUS":
              res += items[i+1](env)
            elif items[i].type == "MINUS":
              res -= items[i+1](env)
          return res

        return eval

      def term(self, items):
        def eval(env):
          for i in range(0,len(items),2):
            if type(items[i](env)) is str:
              return items[i](env)

          res = items[0](env)
          for i in range(1,len(items)-1,2):

            if items[i].type == "DIV":
              res /= items[i+1](env)
            elif items[i].type == "TIMES":
              res *= items[i+1](env)
          return res

        return eval

      def literal(self, items):
        if type(items[1]) is str:
          return items[1]

        def eval(env, x=items[1]):
          a = x(env)
          if type(a) is str:
            return a

          return -a
        return eval

      def id_call(self, items):
        def eval(env, x=items[0].value):
          if x not in env:
            return "Undeclared variable " + x

          return env[x]
        return eval

      def INT(self, items):
        def eval(env, x=items):
          return int(x)
        return eval

  return ValidateTransformer().transform(run_parser(data))

import doctest
doctest.run_docstring_examples(validate, globals())

# Exercise 1

Recall the language for intervals from exercise 2 of TP03. Consider now that the list of intervals is optionally preceded by a `+` or `-` sign that determines whether all intervals should be increasing or decreasing, respectively.

## Exercise 1.1

Write a `lark` grammar to obtain the AST for interval ranges with sign annotations.

In [ ]:
from lark import Lark, Transformer

def run_parser(data):
  """
  Lark parser with doctests.

  >>> run_parser('+[-1.5 ; +2] [3 ; +4.7]')
  Tree(Token('RULE', 'start'), [Tree(Token('RULE', 'sign'), [Token('PLUS', '+')]), Tree(Token('RULE', 'intervals'), [Tree(Token('RULE', 'interval'), [Tree(Token('RULE', 'number'), [Token('FLOAT', '-1.5')]), Tree(Token('RULE', 'number'), [Token('INT', '+2')])]), Tree(Token('RULE', 'interval'), [Tree(Token('RULE', 'number'), [Token('INT', '3')]), Tree(Token('RULE', 'number'), [Token('FLOAT', '+4.7')])])])])

  >>> run_parser("-[3 ; 3.5; 4] [5 ; 12] [48 ; 73.2]")
  Tree(Token('RULE', 'start'), [Tree(Token('RULE', 'sign'), [Token('MINUS', '-')]), Tree(Token('RULE', 'intervals'), [Tree(Token('RULE', 'interval'), [Tree(Token('RULE', 'number'), [Token('INT', '3')]), Tree(Token('RULE', 'number'), [Token('FLOAT', '3.5')]), Tree(Token('RULE', 'number'), [Token('INT', '4')])]), Tree(Token('RULE', 'interval'), [Tree(Token('RULE', 'number'), [Token('INT', '5')]), Tree(Token('RULE', 'number'), [Token('INT', '12')])]), Tree(Token('RULE', 'interval'), [Tree(Token('RULE', 'number'), [Token('INT', '48')]), Tree(Token('RULE', 'number'), [Token('FLOAT', '73.2')])])])])

  >>> run_parser("[3 ; 3.5] [1 ; 10; 12] [48 ; 73.2]")
  Tree(Token('RULE', 'start'), [Tree(Token('RULE', 'intervals'), [Tree(Token('RULE', 'interval'), [Tree(Token('RULE', 'number'), [Token('INT', '3')]), Tree(Token('RULE', 'number'), [Token('FLOAT', '3.5')])]), Tree(Token('RULE', 'interval'), [Tree(Token('RULE', 'number'), [Token('INT', '1')]), Tree(Token('RULE', 'number'), [Token('INT', '10')]), Tree(Token('RULE', 'number'), [Token('INT', '12')])]), Tree(Token('RULE', 'interval'), [Tree(Token('RULE', 'number'), [Token('INT', '48')]), Tree(Token('RULE', 'number'), [Token('FLOAT', '73.2')])])])])

  >>> run_parser("+[3 ; 3.5] [2 ; 1] [0.5 ; -73.2]")
  Tree(Token('RULE', 'start'), [Tree(Token('RULE', 'sign'), [Token('PLUS', '+')]), Tree(Token('RULE', 'intervals'), [Tree(Token('RULE', 'interval'), [Tree(Token('RULE', 'number'), [Token('INT', '3')]), Tree(Token('RULE', 'number'), [Token('FLOAT', '3.5')])]), Tree(Token('RULE', 'interval'), [Tree(Token('RULE', 'number'), [Token('INT', '2')]), Tree(Token('RULE', 'number'), [Token('INT', '1')])]), Tree(Token('RULE', 'interval'), [Tree(Token('RULE', 'number'), [Token('FLOAT', '0.5')]), Tree(Token('RULE', 'number'), [Token('FLOAT', '-73.2')])])])])
  """

  grammar = r"""

  start : sign? intervals
  
  intervals : interval+

  interval : "[" number (";" number)+ "]"
  
  number : INT | FLOAT
  sign : PLUS | MINUS
  
  PLUS : "+"
  MINUS : "-"
  FLOAT : /[+-]?\d+\.\d+/
  INT   : /[+-]?\d+/
  
  %ignore /[ \t\f]+/
  """

  parser = Lark(grammar)
  return parser.parse(data)

import doctest
doctest.run_docstring_examples(run_parser, globals())

## Exercise 1.2

Write a `lark` transformer to validate of intervals (always increasing or always decreasing, according to the sign).

In [ ]:
def validate(data):
  """
  Lark interval validator with doctests.

  >>> print(validate('-[11 ; 0] [0 ; -1; -1.5] [-2 ; -3] [-3 ; -4]'))
  True
  >>> print(validate('-[-200 ; -140; -123] [-100 ; 0] [-1.5 ; +2] [3 ; +4.7]'))
  False
  >>> print(validate("[3 ; 3.5] [5 ; 12] [48 ; 13.2]"))
  True
  >>> print(validate("+[34 ; 36.5]"))
  True
  >>> print(validate("+[3 ; 3.5] [63 ; 80; 12] [98 ; 173.2]"))
  False
  >>> print(validate("[3 ; 3.5] [2 ; 1] [0.5 ; -73.2]"))
  True
  >>> print(validate('+[0 ; 5; 10] [5 ; 10]'))
  False
  """
  class ValidationTransformer(Transformer):
      def start(self, items):
        if len(items) == 1:
          return items[0](0)
        else: 
          return items[1](items[0])
         
      def sign(self, items):
        if items[0].type == "PLUS":
          return 1
        elif items[0].type == "MINUS":
          return 2

      def intervals(self, items):
        def eval(mode):
          for intr in items:
            if not intr(mode):
              return False
          return True
        return eval
      
      def interval(self, items):
        def eval(mode):
          last = items[0]
          for i in range(1, len(items)):
            if mode == 2:
              if last < items[i]:
                return False
            elif mode == 1:  
              if last > items[i]:
                return False
            
            last = items[i]
          return True
        
        return eval

      def number(self, items):
        return float(items[0])

  return ValidationTransformer().transform(run_parser(data))

import doctest
doctest.run_docstring_examples(validate, globals())

# Exercise 2

Recall exercise 1 from TP03 to process propositional logic formulas. We want to extend the logic to allow the encoding of first-order logic formulas, introducing the notion of quantifier and predicate of arity $n$.

First-order logic formulas have thus the following syntax:

- Predicates $A$, $B$, $C$, ...
- Variables $a$, $b$, $c$, ...
- Constants false $⊥$ and true $⊤$
- Predicate application $A(a_1, \ldots, a_n)$ if $A$ has arity $n$
- Conjunction $f_1 ∧ f_2$,
- Disjunction $f_1 ∨ f_2$,
- Negation $¬ f_1$,
- Implication $f_1 → f_2$
- Equivalence $f_1 ↔︎ f_2$
- Universal quantification $∀ a . f_1$
- Existential quantification $∃ a . f_1$

Variables are introduced by quantifications, while predicates must be declared externally and have their arity assigned. For instance, the following is a valid first-order formula declaration:
```
A : 1
R : 2
∀ a . A(a) → ∃ b . R(a,b)
```

## Exercise 2.1

Write an attribute grammar for this language:
- Write the context-free grammar
- Identifying the synthesized and inherited attributes for each grammar variable
- Define the evaluation rules for each attribute

## Exercise 2.2

Write a `lark` grammar to obtain the AST for first-order logic formula declarations.

In [ ]:
from lark import Lark, Transformer

def run_parser(data):
  """
  Lark parser with doctests.

  >>> run_parser("A : 0 B: 0 A() ∧ ¬B()")
  Tree(Token('RULE', 'start'), [Tree(Token('RULE', 'decls'), [Tree(Token('RULE', 'decl'), [Token('PRD', 'A'), Token('INT', '0')]), Tree(Token('RULE', 'decl'), [Token('PRD', 'B'), Token('INT', '0')])]), Tree(Token('RULE', 'conj'), [Tree(Token('RULE', 'application'), [Token('PRD', 'A'), Tree(Token('RULE', 'vars'), [])]), Token('CONJ', '∧'), Tree(Token('RULE', 'neg'), [Token('NEG', '¬'), Tree(Token('RULE', 'application'), [Token('PRD', 'B'), Tree(Token('RULE', 'vars'), [])])])])])

  >>> run_parser("A : 1 ∃ b . A(b)")
  Tree(Token('RULE', 'start'), [Tree(Token('RULE', 'decls'), [Tree(Token('RULE', 'decl'), [Token('PRD', 'A'), Token('INT', '1')])]), Tree(Token('RULE', 'formula'), [Token('SOME', '∃'), Tree(Token('RULE', 'bnd'), [Token('ID', 'b')]), Tree(Token('RULE', 'application'), [Token('PRD', 'A'), Tree(Token('RULE', 'vars'), [Token('ID', 'b')])])])])

  >>> run_parser("A : 1 R : 2 ∀ a . (A(a) → ∃ b . R(a,b))")
  Tree(Token('RULE', 'start'), [Tree(Token('RULE', 'decls'), [Tree(Token('RULE', 'decl'), [Token('PRD', 'A'), Token('INT', '1')]), Tree(Token('RULE', 'decl'), [Token('PRD', 'R'), Token('INT', '2')])]), Tree(Token('RULE', 'formula'), [Token('ALL', '∀'), Tree(Token('RULE', 'bnd'), [Token('ID', 'a')]), Tree(Token('RULE', 'implies'), [Tree(Token('RULE', 'application'), [Token('PRD', 'A'), Tree(Token('RULE', 'vars'), [Token('ID', 'a')])]), Token('IMPLY', '→'), Tree(Token('RULE', 'formula'), [Token('SOME', '∃'), Tree(Token('RULE', 'bnd'), [Token('ID', 'b')]), Tree(Token('RULE', 'application'), [Token('PRD', 'R'), Tree(Token('RULE', 'vars'), [Token('ID', 'a'), Token('ID', 'b')])])])])])])

  >>> run_parser("⊤")
  Tree(Token('RULE', 'start'), [Tree(Token('RULE', 'decls'), []), Tree(Token('RULE', 'constant'), [Token('TRUE', '⊤')])])
  """
  grammar = r"""
    start : decls formula

    decls : decl*

    decl : PRD ":" INT

    ?formula : SOME bnd "." formula
            | ALL bnd "." formula
            | equiv
    
    bnd : ID

    exm : application
    
    ?equiv : equiv EQUIV implies
            | implies

    ?implies: implies IMPLY conj
        | conj
    
    ?conj: conj CONJ disj
        | disj

    ?disj: disj DISJ neg
        | neg

    ?neg: NEG term
        | term

    ?term: constant | paren | application | formula

    application : PRD "(" vars ")"

    vars : (ID(","ID)*)?
    ?paren: "(" formula ")"
    var: ID
    constant: TRUE | FALSE

    CONJ: "∧"
    IMPLY: "→"
    EQUIV: "↔︎"
    DISJ: "∨"
    NEG: "¬"
    TRUE: "⊤"
    FALSE: "⊥"
    ALL: "∀" 
    SOME: "∃"

    ID : /\w+/
    PRD : /[A-Z]/
    INT : /[0-9]/

    %ignore /[ \t\f]+/

  """

  parser = Lark(grammar)
  return parser.parse(data)

import doctest
doctest.run_docstring_examples(run_parser, globals())

## Exercise 2.3

Write a `lark` transformer to validate the declaration of first-order logic formulas. Namely:
- A variable $a$ may only occur if it has been introduced by a quantifier
- A predicate $A$ may only occur if it has been introduced by a quantifier
- A predicate application $A(a_1,\ldots,a_n)$ may conform to the arity declared for $A$

In [ ]:
def validate(data):
  """
  Lark validator with doctests.

  >>> print(validate("A : 0 B: 0 A() ∧ ¬B()"))
  None

  >>> print(validate("A : 0 B: 0 X() ∧ ¬B()"))
  Undeclared predicate called X

  >>> print(validate("A : 1 B: 0 A() ∧ ¬B()"))
  Invalid arity in call to A

  >>> print(validate("A : 1 B: 0 A(x) ∧ ¬B()"))
  Undeclared variable x

  >>> print(validate("A : 1 B: 0 ∀ x . A(x) ∧ ¬B()"))
  None

  >>> print(validate("A : 1 R : 2 ∀ a . (A(a) → ∃ b . R(a,b))"))
  None

  >>> print(validate("A : 1 R : 2 ∀ a . (A(a) → ∃ b . R(a,b))"))
  None

  >>> print(validate("A : 1 R : 2 ∀ a . (A(a) → ∃ b . R(a,b,c))"))
  Undeclared variable c

  >>> print(validate("A : 1 R : 2 ∀ a . (A(a) → ∃ b . R(a,b,a))"))
  Invalid arity in call to R

  >>> print(validate("A : 1 R : 2 ∀ a . (A(a) → ∃ b . R(b))"))
  Invalid arity in call to R

  >>> print(validate("⊤"))
  None

  >>> print(validate("A : 1 (∀ a . A(a)) ∧ A(a)"))
  Undeclared variable a
  """
  class ValidateTransformer(Transformer):
      def __init__(self):
        self.env = {}
        self.bn = set()

      def start(self, items):
        for it in items:
          if type(it) is str:
              return it
        if len(items) == 1:
            return items[0]
        else:
            return items[1]
      
      def decls(self, items):
        for it in items:
          if type(it) is str:
              return it
        pass
        
      def bnd(self, items):
        for it in items:
          if type(it) is str:
              return it
        for i in items:
            self.bn.add(i.value)
        return self.bn
      
      def equiv(self, items):
        for it in items:
          if type(it) is str:
              return it
        return None

      def implies(self, items):
        for it in items:
          if type(it) is str:
              return it
        return None

      def disj(self, items):
        for it in items:
          if type(it) is str:
              return it
        return None

      def conj(self, items):
        for it in items:
          if type(it) is str:
              return it
        return None

      def neg(self, items):
          for it in items:
              if type(it) is str:
                  return it
          return None

      def term(self, items):
          for it in items:
              if type(it) is str:
                  return it
          return None
      
      def application(self, items):
          for it in items:
              if type(it) is str:
                  return it
              
          name = items[0].value
          args = items[1]
          
          if name not in self.env:
              return f"Undeclared predicate called {items[0]}"
                
          if len(args) != self.env[name]:
              return f"Invalid arity in call to {items[0]}"
          return None
      
      def vars(self, items):
          lista = []
          for it in items:
              if type(it) is str:
                  return it
          for var in items:
              name = var.value
              if name not in self.bn:
                  return f"Undeclared variable {name}"
              lista.append(name)
          return lista

      def formula(self, items):
          for it in items:
            if type(it) is str:
              return it
          chave_a_remover = list(items[1])[0]
          if items[0] == "∀" or items[0] == "∃":
            self.bn.remove(chave_a_remover)
          pass
         
      def decl(self, items):
        for prd in items:
          if prd[0] in self.env:
            return "Predicate " + prd[0] + " already declared"
          arity = int(items[1].value)
          self.env[items[0].value] = arity
        return self.env
      
      def paren(self, items):
        pass

      def constant(self, items):
        pass

  return ValidateTransformer().transform(run_parser(data))

import doctest
doctest.run_docstring_examples(validate, globals())

# Exercise 3

Recall the tiny functional language from exercise 5 of TP03. Now, extend its support to also allow variables, declared with a given type. So a formula can be:

- Integer and Boolean literals
- A previously declared variable
- Binary operators (`+`, `*`, `<`, `and`, `or`)
- Conditional expressions (`if ... then ... else ...`)

For instance, the following is a valid expression in this language:
```
x : int
y : bool
if 2 > 3 or y then 10 else -1*x
```

## Exercise 3.1

Write a `lark` grammar to obtain the AST for this tiny language.

In [ ]:
from lark import Lark, Transformer
def run_parser(example):
    """
    Lark parser with doctests.

    >>> run_parser("x : int; p : bool; if 2 > x or p > 10 then 10 else x*10")
    Tree(Token('RULE', 'start'), [Tree(Token('RULE', 'decls'), [Tree(Token('RULE', 'decl'), [Token('ID', 'x'), Token('INT', 'int')]), Tree(Token('RULE', 'decl'), [Token('ID', 'p'), Token('BOOL', 'bool')])]), Tree(Token('RULE', 'ifexpr'), [Token('IF', 'if'), Tree(Token('RULE', 'boolexpr'), [Tree(Token('RULE', 'booleterm'), [Tree(Token('RULE', 'boolfactor'), [Tree(Token('RULE', 'factor'), [Token('INT', '2')]), Token('GT', '>'), Tree(Token('RULE', 'factor'), [Token('ID', 'x')])]), Token('OR', 'or'), Tree(Token('RULE', 'boolterm'), [Tree(Token('RULE', 'boolfactor'), [Token('ID', 'p'), Token('GT', '>'), Tree(Token('RULE', 'factor'), [Token('INT', '10')])])])])]), Token('THEN', 'then'), Tree(Token('RULE', 'ifexpr'), [Token('INT', '10')]), Token('ELSE', 'else'), Tree(Token('RULE', 'ifexpr'), [Tree(Token('RULE', 'expr'), [Tree(Token('RULE', 'term'), [Tree(Token('RULE', 'factor'), [Token('ID', 'x')]), Token('TIMES', '*'), Tree(Token('RULE', 'factor'), [Token('INT', '10')])])])])])])
    """
    grammar=r"""

    start: decls ifexpr

    decls: decl*

    decl: ID ":" (int | bool) ";"

    int : "int"
    bool : "bool"

    ?ifexpr : "if" boolexpr "then" ifexpr "else" ifexpr | expr

    ?expr : expr (PLUS|MINUS) term | term

    ?term : term (TIMES|DIV) factor | factor

    ?boolexpr : boolexpr OR boolterm | boolterm
    ?boolterm : boolterm AND boolfactor | boolfactor
    ?boolfactor : boolconst | expr (GT|LT) expr

    boolconst : "false" | "true"
    factor : constant | variable | "(" ifexpr ")"

    constant : INT
    variable : ID

    GT : ">"
    LT : "<"
    PLUS : "+"
    MINUS : "-"
    DIV : "/"
    TIMES : "*"
    INT : /\d+/
    ID : /\w+/
    OR : "or"
    AND : "and"

    %ignore /[ \t]+/
    """

    parser = Lark(grammar)
    return parser.parse(example)

import doctest
doctest.run_docstring_examples(run_parser, globals())

#print(parser.parse(example).pretty())

## Exercise 3.2

Write a `lark` transformer to type check this language.

For instance, the following expression type checks:
```
x : int
p : bool
if 2 > x or p then 10 else -x*10
```

But the following (syntactically correct) should throw a type error:

```
p : bool
if 2 > p then 10 else -1*10
```
```
x : int
p : bool
if 2 > 3 or 4 < 10 then x else p
```
```
p : bool
if 2 > 3 and 3 + p then 10 < 20 else True
```


In [ ]:
def stats(data):
  """
  Lark statististics collector with doctests.
  >>> print(stats("x : int; p : bool; if 2 > x or p > 10 then 10 else x*10"))
  True

  >>> print(stats("x : bool; p : bool; if 2 > x or p > 10 then 10 else x*10"))
  Error in boolean expression
  """

  class ValidateTransformer(Transformer):
      def __init__(self):
          self.symbol = {}

      def start(self, items):
        
        return items[1]

      def decls(self, items):
        pass

      def boolexpr(self, items):

        if items[0] is bool and items[2] is bool:
          return bool
        return "Error in boolean expression"

      def boolfactor(self, items):
        print(f"AQUI MEu AMIGO {items}")
        if items[0] is int and items[2] is int:
          return bool
        return "Error in boolean factor"

      def variable(self, items):
        return self.symbol[items[0]]

      def constant(self, items):
        return int

      def boolconst(self, items):
        return bool

      def decl(self, items):
        self.symbol[items[0].value] = items[1]

      def int(self, items):
        return int

      def bool(self, items):
        return bool


  return ValidateTransformer().transform(run_parser(data))


import doctest
doctest.run_docstring_examples(stats, globals())



# Exercise 4

Consider now that the previous language was extended with the definition of functions, which can also contain declared variables. Consider the following rules:
- A function and a variable can only be called if declared previously
- The scope of variables declared inside functions is always local
- A function can redefine a variable
- A function can use variables from the global scope

For instance, the following is correct code:
```
let g = 20;
let f(x) = let y = x * 2; y * g;
f(20) + 10
```

```
let f(x) = let y = x * 2; y * 10;
let x = 2; f(x)+10
```

```
let x = 2;
let f(x) = let y = x * 2; y * 10;
f(x) + 10
```


But the following is invalid:
```
let f(x) = let y = x * 2; y * 10;
f(x) + 10
```

## Exercise 4.1

Write a Lark grammar to parse this language.

In [ ]:
grammar=r""" 
    start: decls
    decls: decl*
    
    ?decl: 'let' VAR '=' expr ';' | 'let' FUNC '=' expr ';'

    ?expr: INT
        | VAR
        | FUNC
        | expr '+' expr
        | expr '-' expr
        | expr '*' expr
        | expr '/' expr

    FUNC : ID '(' VAR (',' VAR)* ')' '=' expr ';'
    ID : /[a-z]+/
    VAR : /[a-z]/
    INT : /\d+/
"""

## Exercise 4.2

Use Lark AST traversals to check the scopes of the variables.

## Exercise 4.3

Write a Lark `Visitor` to rename local calls with their scope name. For instance, a variable `x` inside a function `f`, should be renamed as `f_x`.

## Exercise 4.4

Write Lark `Transformer` to evaluate the value of expressions in this language.